# Capstone · Phase 4 上机：因果实验设计与验证

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据集）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 在**真实RCT数据**（NSW职业培训实验）上完成DoWhy四步因果分析（假设->识别->估计->反驳）
2. 用**DML双重机器学习**和**因果森林**估计异质因果效应（CATE）
3. 用**CUPED**利用前实验协变量降低实验方差
4. 用**自定义BaseMetric**（deepeval fallback）评估Agent输出中因果证据使用质量
5. 整合技能3（因果推断）+技能5（Agent评估），回答「营销Agent的干预真的有效吗？」

## 真实数据
- **NSW真实RCT**（`causaldata`包）：`treat`=营销干预, `re78`=转化, `re75`=基线
- **真实库**：DoWhy + econml + causaldata + 自定义BaseMetric

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata dowhy econml scikit-learn statsmodels -q

## 1. 数据集背景与营销映射

**NSW数据集**：NSW职业培训示范实验的真实数据（Dehejia & Wahba 1999），因果推断最经典的真实教学数据集。

| NSW变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到营销干预（优惠券/广告/Agent系统） | 处理 T |
| `re78` | 转化率/GMV/客单价 | 结果 Y |
| `re75` | 基线消费/历史转化率 | 前实验协变量（CUPED用） |
| `age`, `educ`, `black`, `hisp`, `marr`, `nodegree`, `re74` | 用户画像特征 | 协变量 X（混杂） |

**核心因果问题**：营销干预（treat）对转化（re78）的真实因果效应（ATE）是多少？哪些用户群体效应更大（CATE）？

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import dowhy
from dowhy import CausalModel
from causaldata import nsw_mixtape
from econml.dml import LinearDML, CausalForestDML
from sklearn.ensemble import RandomForestRegressor

# 评估库（deepeval fallback）
try:
    from deepeval.metrics import BaseMetric
    HAS_DEEPEVAL = True
except ImportError:
    HAS_DEEPEVAL = False
    print('deepeval未安装，使用自定义BaseMetric fallback')

## TODO 1-2：加载真实数据 + 朴素估计

先加载数据、检查协变量均衡性，再算朴素均值差（有偏估计）。

In [ ]:
# TODO 1：加载真实NSW数据，检查协变量均衡性
# 提示：nsw_mixtape.load_pandas().data
# 要求：加载为df，打印形状，按treat分组对比至少3个协变量均值

# ===== 你的代码 =====
df = None  # TODO: 替换这行
# ====================

print(f'数据形状: {df.shape}')
print(f'处理组: {len(df[df["treat"]==1])}, 对照组: {len(df[df["treat"]==0])}')

In [ ]:
# TODO 2：朴素估计 -- 直接算处理组-对照组re78均值差（有偏）
# 提示：df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
# 要求：打印朴素ATE

# ===== 你的代码 =====
naive_ate = None  # TODO
# ====================

print(f'朴素估计 ATE = {naive_ate:.2f}')
print('⚠️ 这个估计有偏！协变量分布不均（见TODO1）')

## 2. 因果图（DAG）与混杂分析

NSW场景的因果结构（简化）：

```
age, educ ──> treat ──> re78
     │            ▲
     └────────────┘   (后门路径)
re74, re75 ──> treat ──> re78
     │            ▲
     └────────────┘   (后门路径)
```

- **混杂因素**：age/educ/re74/re75同时影响treat和re78
- **后门路径**：treat ← age/educ/re74/re75 → re78（创造虚假相关）
- **后门准则**：控制这些协变量即可识别treat→re78的因果效应

**营销类比**：收到优惠券的用户可能本来就是高活跃用户（自选择），朴素均值差混淆了「用户特征」和「优惠券效果」。

## TODO 3：DoWhy四步因果分析

用DoWhy完成建模->识别->估计->反驳四步因果分析。

In [ ]:
# TODO 3：DoWhy四步因果分析（建模->识别->估计->反驳）
# 提示：
#   model = CausalModel(data=df, treatment='treat', outcome='re78', common_causes=common_causes)
#   identified = model.identify_effect()
#   estimate = model.estimate_effect(identified, method_name='backdoor.linear_regression')
#   refutation = model.refute_estimate(identified, estimate, 'placebo_treatment_refuter')
# 要求：打印ATE、安慰剂检验结果

common_causes = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']

# ===== 你的代码 =====

# ====================

print(f'DoWhy后门调整 ATE = {causal_estimate.value:.2f}')
print(f'安慰剂检验: {refutation}')

## 3. CUPED方差缩减

CUPED（Deng et al. 2013, Microsoft KDD）利用前实验协变量调整结果变量，缩小方差、提升检测灵敏度。

$$Y_{adj} = Y - \theta \cdot (X_{pre} - \bar{X}_{pre}), \quad \theta = \frac{\text{Cov}(Y, X_{pre})}{\text{Var}(X_{pre})}$$

在NSW中，用 `re75`（前一年收入）调整 `re78`（结果收入）。营销映射：用基线消费调整转化率。

In [ ]:
# TODO 4：CUPED方差缩减 -- 用re75调整re78
# 提示：theta = np.cov(Y, X_pre)[0,1] / np.var(X_pre)
#       Y_adj = Y - theta * (X_pre - np.mean(X_pre))
# 要求：打印CUPED ATE和方差缩减比例

Y = df['re78'].values
X_pre = df['re75'].values
T = df['treat'].values

# ===== 你的代码 =====

# ====================

print(f'朴素 ATE = {naive_ate:.2f}')
print(f'CUPED ATE = {cuped_ate:.2f}')
print(f'方差缩减: {var_reduction:.2%}')

## 4. DML双重机器学习

DML（Chernozhukov et al. 2018）用ML模型估计nuisance functions，再用残差化Y和T估计因果效应。相比线性回归，DML在高维协变量和非线性关系下更稳健。

用 `econml.dml.LinearDML` 估计ATE和CATE（异质因果效应）。

In [ ]:
# TODO 5：DML双重机器学习 -- 用econml估计ATE和CATE
# 提示：
#   dml = LinearDML(model_y=RandomForestRegressor(n_estimators=50, random_state=42),
#                   model_t=RandomForestRegressor(n_estimators=50, random_state=42),
#                   discrete_treatment=True, random_state=42)
#   dml.fit(Y, T, X=X)
#   dml_ate = dml.ate(X=X)
#   dml_ci = dml.ate_interval(X=X, alpha=0.05)
# 要求：打印DML ATE、95%CI、按年龄分组的CATE

covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
X = df[covariates].values

# ===== 你的代码 =====

# ====================

print(f'DML ATE = {dml_ate:.2f}, 95% CI: [{dml_ci[0]:.2f}, {dml_ci[1]:.2f}]')
print(f'CATE (young): {cate_young:.2f}, CATE (older): {cate_old:.2f}')

## 5. 因果森林

因果森林（Wager & Athey 2018）用随机森林结构估计异质因果效应（CATE），能自动发现协变量交互效应。

用 `econml.dml.CausalForestDML` 估计CATE，对比DML的异质效应发现。

In [ ]:
# TODO 6：因果森林 -- 用econml估计CATE
# 提示：CausalForestDML(model_y=..., model_t=..., discrete_treatment=True, n_estimators=100, random_state=42)
# 要求：打印因果森林ATE、95%CI，与DML对比

# ===== 你的代码 =====

# ====================

print(f'因果森林 ATE = {cf_ate:.2f}, 95% CI: [{cf_ci[0]:.2f}, {cf_ci[1]:.2f}]')
print(f'DML ATE = {dml_ate:.2f}, 因果森林 ATE = {cf_ate:.2f}')

## 6. Agent因果证据评估（整合技能5 Day 3）

Phase 3的营销Agent系统会输出策略文本。本步评估Agent输出中**因果证据使用质量**：
- 是否引用了ATE/CATE数值？
- 是否提及混杂因素控制？
- 是否提及稳健性检验？
- 是否避免因果过度外推？

用自定义BaseMetric（deepeval fallback，无API key时规则评估）。

In [ ]:
# TODO 7：Agent因果证据评估 -- 自定义BaseMetric
# 提示：实现CausalEvidenceMetric类，评估4个维度（ATE引用/混杂控制/稳健性检验/避免过度外推）
# 要求：用good_output和bad_output测试，打印评分

# ===== 你的代码 =====

# ====================

good_output = '''基于DoWhy因果分析，营销干预的ATE为1676（95% CI: [608, 3271]）。
我们通过后门调整控制了年龄、教育、前期收入等混杂因素。
安慰剂检验p值为0.98，新效应接近0，说明估计稳健。
建议在25岁以上用户群体中推广（CATE更高），但需注意可忽略性假设的局限。'''

bad_output = '''营销干预必定有效！转化率绝对提升了100%。
因果证明了优惠券的效果，不需要考虑混杂因素。'''

metric = CausalEvidenceMetric(threshold=0.7)
print(f'Good output score: {metric.measure(good_output):.2f}')
print(f'Bad output score: {metric.measure(bad_output):.2f}')

## 7. 反思与前沿

### 反思问题
1. 朴素ATE vs DoWhy后门ATE vs DML ATE的差异来自哪里？哪个最可信？
2. DML和因果森林的CATE估计是否一致？哪个用户群体获益最大？
3. CUPED调整后方差降低了多少？这对实验设计有什么启示？
4. 安慰剂检验和随机混杂检验的p值是否支持你的因果估计？
5. 如果NSW数据里有个**没观测到的混杂**（如「个人上进心」），你的估计还可靠吗？

### 2026前沿
- **DML双重机器学习**：高维协变量下的无偏因果估计
- **CUPED**：方差缩减技术，等效提升样本量
- **因果森林**：自动发现异质因果效应
- **Uplift/增量建模**：按因果效应排序优先干预「可被说服」用户
- **MAB多臂老虎机**：多干预方案的自适应选择
- **贝叶斯因果推断**：小样本下的不确定性量化

### 整合性
本Phase整合了技能3（因果推断Day1-5）和技能5（Day3评估）：
- 技能3提供因果推断方法论（DoWhy四步+DML+因果森林+CUPED）
- 技能5提供Agent评估方法论（BaseMetric+LLM-as-a-judge）
- Phase 4用因果推断评估Agent系统的效果，用Agent评估验证Agent是否正确使用因果证据